# Daily Exercise: Heart Disease Classification

**Course:** Developers Institute  **Week 5 - Day 2**  
**Author:** Alex Goldbaum

Goal: build and compare three classification models — **Logistic Regression**,
**SVM**, and **XGBoost** — on the Heart Disease Prediction Dataset, each both
with manual hyperparameters and tuned via `GridSearchCV`. We report accuracy,
precision, recall, F1 and ROC-AUC on a held-out test set so we can pick the
winner on evidence.

Dataset: **UCI Cleveland Heart Disease** (303 patients, 13 clinical features +
binary target).


## Setup


In [ ]:
%pip install -qU xgboost


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay,
)
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


## Exercise 1: Exploratory Data Analysis

Load the data, inspect, handle missing values, then split into train/test.


In [ ]:
URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
COLUMNS = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num',
]

df = pd.read_csv(URL, header=None, names=COLUMNS, na_values='?')
print('Shape:', df.shape)
df.head()


In [ ]:
# Missing values and dtypes
print('Missing values per column:')
print(df.isna().sum())
print('\nDtypes:')
print(df.dtypes)


In [ ]:
# Build the binary target: original `num` is 0–4 (0 = no disease, 1–4 = disease).
# Following the standard convention we collapse it to a binary target.
df['target'] = (df['num'] > 0).astype(int)
df = df.drop(columns=['num'])

# Impute missing values (only `ca` and `thal` have a handful) with the median
df = df.fillna(df.median(numeric_only=True))

print('After cleaning — missing values:', df.isna().sum().sum())
print('Class distribution:')
print(df['target'].value_counts().rename({0: 'No disease', 1: 'Disease'}))


In [ ]:
# Quick visual EDA
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x='target', data=df, palette=['steelblue', 'tomato'], ax=axes[0])
axes[0].set_xticklabels(['No disease', 'Disease'])
axes[0].set_title('Class distribution', fontweight='bold')

sns.histplot(data=df, x='age', hue='target', kde=True, bins=20, ax=axes[1])
axes[1].set_title('Age distribution by class', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Correlation with target
corr = df.corr()['target'].drop('target').sort_values()

plt.figure(figsize=(8, 5))
colors = ['tomato' if v < 0 else 'steelblue' for v in corr.values]
plt.barh(corr.index, corr.values, color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Pearson correlation of each feature with target', fontweight='bold')
plt.xlabel('Correlation')
plt.tight_layout()
plt.show()


In [ ]:
# Train/test split (stratified)
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]} ({y_train.mean()*100:.1f}% positive)')
print(f'Test : {X_test.shape[0]} ({y_test.mean()*100:.1f}% positive)')


**EDA notes.** The dataset is roughly balanced (~45% positive). The strongest
positive correlations with disease are `cp` (chest pain type), `exang` (exercise
induced angina), `oldpeak` and `ca`. The strongest negative correlation is
`thalach` (max heart rate achieved) — patients with higher peak heart rates tend
to be healthier in this cohort. We stratify the split to preserve the class ratio.


## Helper — uniform evaluation across all models

All seven exercises end up reporting the same set of metrics, so we factor this
into a tiny helper and collect every run in a results dataframe.


In [ ]:
results = []  # populated by every exercise below

def report(name, model, X_te, y_te, best_params=None):
    """Score `model` on `X_te`/`y_te`, print and append to `results`."""
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_te)
    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_te, pred),
        'Precision': precision_score(y_te, pred),
        'Recall': recall_score(y_te, pred),
        'F1': f1_score(y_te, pred),
        'ROC-AUC': roc_auc_score(y_te, proba),
    }
    if best_params is not None:
        metrics['Best params'] = best_params
    results.append(metrics)

    print(f'=== {name} ===')
    for k, v in metrics.items():
        if k in ('Model', 'Best params'):
            print(f'  {k}: {v}')
        else:
            print(f'  {k}: {v:.4f}')
    print()
    return metrics


## Exercise 2: Logistic Regression without Grid Search

Default-ish hyperparameters with `StandardScaler` inside a `Pipeline` so the
scaler never sees test data.


In [ ]:
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
lr_pipe.fit(X_train, y_train)

report('Logistic Regression (no grid)', lr_pipe, X_test, y_test)


## Exercise 3: Logistic Regression with Grid Search

Tune `C` (inverse regularization strength) and `penalty`. We use the `liblinear`
solver because it supports both L1 and L2 penalties.


In [ ]:
lr_grid_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000, solver='liblinear', random_state=RANDOM_STATE)),
])

lr_param_grid = {
    'clf__C': [0.01, 0.1, 1, 10, 100],
    'clf__penalty': ['l1', 'l2'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
lr_search = GridSearchCV(lr_grid_pipe, lr_param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
lr_search.fit(X_train, y_train)

print(f'Best CV ROC-AUC: {lr_search.best_score_:.4f}')
report('Logistic Regression (grid)', lr_search.best_estimator_, X_test, y_test, lr_search.best_params_)


## Exercise 4: SVM without Grid Search

Manual choice: RBF kernel with `C=1` and `gamma='scale'`. SVM is scale-sensitive
so we keep `StandardScaler` in front. We set `probability=True` so we can report
ROC-AUC.


In [ ]:
svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=RANDOM_STATE)),
])
svm_pipe.fit(X_train, y_train)

report('SVM (no grid)', svm_pipe, X_test, y_test)


## Exercise 5: SVM with Grid Search

Tune `C`, `kernel` and `gamma`. The grid is intentionally small so it runs in
seconds on Colab.


In [ ]:
svm_grid_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(probability=True, random_state=RANDOM_STATE)),
])

svm_param_grid = {
    'clf__C': [0.1, 1, 10],
    'clf__kernel': ['linear', 'rbf'],
    'clf__gamma': ['scale', 0.01, 0.1],
}

svm_search = GridSearchCV(svm_grid_pipe, svm_param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
svm_search.fit(X_train, y_train)

print(f'Best CV ROC-AUC: {svm_search.best_score_:.4f}')
report('SVM (grid)', svm_search.best_estimator_, X_test, y_test, svm_search.best_params_)


## Exercise 6: XGBoost without Grid Search

Manual hyperparameters that are reasonable defaults for a small, tabular dataset:
modest `n_estimators`, moderate `learning_rate`, shallow trees to avoid overfit.
XGBoost is scale-invariant so no scaler is needed.


In [ ]:
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb.fit(X_train, y_train)

report('XGBoost (no grid)', xgb, X_test, y_test)


## Exercise 7: XGBoost with Grid Search

Tune `learning_rate`, `n_estimators`, `max_depth`. Grid kept small.


In [ ]:
xgb_param_grid = {
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 400],
    'max_depth': [2, 3, 5],
}

xgb_base = XGBClassifier(
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_search = GridSearchCV(xgb_base, xgb_param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
xgb_search.fit(X_train, y_train)

print(f'Best CV ROC-AUC: {xgb_search.best_score_:.4f}')
report('XGBoost (grid)', xgb_search.best_estimator_, X_test, y_test, xgb_search.best_params_)


## Comparison of all seven runs


In [ ]:
results_df = pd.DataFrame(results)
display_cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
results_df[display_cols].round(4)


In [ ]:
# Bar chart: ROC-AUC and F1 side by side
comp = results_df.set_index('Model')[['F1', 'ROC-AUC']]
ax = comp.plot(kind='barh', figsize=(10, 6), color=['steelblue', 'seagreen'], edgecolor='white')
plt.title('Model comparison — F1 and ROC-AUC on the test set', fontweight='bold')
plt.xlim(0, 1)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3)
plt.tight_layout()
plt.show()


In [ ]:
# Confusion matrix for the best model (highest ROC-AUC)
best_idx = results_df['ROC-AUC'].idxmax()
best_name = results_df.loc[best_idx, 'Model']
print(f'Best model by ROC-AUC: {best_name}')

# Re-grab the actual estimator
model_lookup = {
    'Logistic Regression (no grid)': lr_pipe,
    'Logistic Regression (grid)': lr_search.best_estimator_,
    'SVM (no grid)': svm_pipe,
    'SVM (grid)': svm_search.best_estimator_,
    'XGBoost (no grid)': xgb,
    'XGBoost (grid)': xgb_search.best_estimator_,
}
best_model = model_lookup[best_name]

cm = confusion_matrix(y_test, best_model.predict(X_test))
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['No disease', 'Disease']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion matrix — {best_name}', fontweight='bold')
plt.tight_layout()
plt.show()

print(classification_report(y_test, best_model.predict(X_test), target_names=['No disease', 'Disease']))


## Analysis report

**1. Which model wins?** Decision criterion: highest ROC-AUC on the held-out test
set, with F1 as a secondary tiebreaker. The leaderboard above (and the bar plot)
make this explicit.

**2. Effect of GridSearchCV.** For Logistic Regression and SVM, grid search
typically produces a small but real lift over the manual baseline — mainly by
tuning `C` away from the default. For XGBoost, the manual baseline is already
competitive because the chosen hyperparameters (modest trees, mid learning rate)
are sensible for a small tabular dataset.

**3. Best-tuned hyperparameters.** Inspect `results_df['Best params']` to see what
GridSearchCV chose. Common patterns on this dataset:
- LR: L2 penalty with moderate `C` (around 1) — strong regularization is not
  needed, but L1 occasionally wins by zeroing out weak features.
- SVM: RBF kernel with `gamma='scale'` and `C ≈ 1`; linear kernels can also be
  competitive on this dataset because of the strong linear signals from `cp`,
  `exang`, `oldpeak`.
- XGBoost: shallow trees (`max_depth=2–3`) and ~100–200 estimators usually win,
  because the dataset is small and deeper trees overfit fast.

**4. Production considerations.**
- Health datasets are imbalanced in the wild — pick the operating threshold to
  match the cost of a missed diagnosis (false negative) vs an unnecessary follow-up
  (false positive); do not rely on the default 0.5.
- Calibrate the model probabilities (Platt scaling / isotonic regression) so that
  downstream clinical workflows can trust the probability number.
- Audit fairness across age, sex, and ethnicity subgroups.
- Monitor for drift after deployment — clinical practice and patient populations
  change over time.

**5. Limitations.** The Cleveland dataset has only 303 patients, so test-set
metrics carry meaningful variance. A serious deployment would re-validate on a
much larger, modern cohort.
